# Config & dataset overview

Read-only monitor. Re-run top to bottom after changing any config or ingesting data.
Nothing here writes to disk or triggers a pipeline.

1. Config discovery (every `*.json` under `src/`)
2. Bronze datasets: what the configs declare vs. what is on disk
3. On-disk contents per dataset (images / labels per split)
4. Config health checks
5. Data layers on disk (`data/<task>/<layer>`)
6. Task registry + silver DAGs

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR = "data"  # every dataset_location is expected to live under this
IMG_EXT = {".jpg", ".jpeg", ".png"}
print("repo root:", ROOT)

## 1. Config discovery

In [ ]:
def discover_configs(root):
    found = []
    for path in sorted((root / "src").rglob("*.json")):
        if "__pycache__" in path.parts:
            continue
        try:
            cfg = json.loads(path.read_text(encoding="utf-8"))
        except json.JSONDecodeError as exc:
            found.append({"path": path, "kind": "invalid", "cfg": {}, "error": str(exc)})
            continue
        cfg = cfg if isinstance(cfg, dict) else {}
        datasets = cfg.get("datasets", [])
        if any(isinstance(d, dict) and "chain" in d for d in datasets):
            kind = "silver"
        elif datasets:
            kind = "bronze"
        else:
            kind = "other"
        found.append({"path": path, "kind": kind, "cfg": cfg})
    return found


CONFIGS = discover_configs(ROOT)
display(pd.DataFrame([
    {
        "file": str(c["path"].relative_to(ROOT)),
        "kind": c["kind"],
        "task_name": c["cfg"].get("task_name"),
        "datasets": len(c["cfg"].get("datasets", [])),
        "error": c.get("error", ""),
    }
    for c in CONFIGS
]))

## 2. Bronze datasets (config vs. disk)

In [ ]:
def resolve(location):
    return (ROOT / location).resolve()


def find_on_disk(location):
    """Config path if it exists, else the same path re-based under data/ if that exists."""
    if not location:
        return None
    path = resolve(location)
    if path.exists():
        return path
    parts = Path(location).parts
    if parts and parts[0] != DATA_DIR:
        alt = resolve(str(Path(DATA_DIR, *parts[1:])))
        if alt.exists():
            return alt
    return None


rows = []
for c in CONFIGS:
    if c["kind"] != "bronze":
        continue
    for d in c["cfg"]["datasets"]:
        p = d.get("params", {})
        if d.get("dataset_type") == "roboflow":
            ref = f"{p.get('workspace')}/{p.get('project')}"
        else:
            ref = p.get("dataset_name")
        location = p.get("dataset_location", "")
        found = find_on_disk(location)
        rows.append({
            "config": c["path"].parent.name,
            "task_name": c["cfg"].get("task_name"),
            "source": d.get("dataset_type"),
            "dataset": p.get("dataset_name"),
            "ref": ref,
            "version": p.get("version"),
            "location": location,
            "on_disk": resolve(location).exists() if location else False,
            "found_at": str(found.relative_to(ROOT)) if found else None,
        })

bronze = pd.DataFrame(rows)
display(bronze)

## 3. On-disk contents

In [ ]:
def inspect_dataset(path):
    info = {"files": 0, "classes": None, "splits": {}}
    if not path.exists():
        return info
    info["files"] = sum(1 for f in path.rglob("*") if f.is_file())
    data_yaml = path / "data.yaml"
    if data_yaml.exists():
        names = yaml.safe_load(data_yaml.read_text(encoding="utf-8")).get("names")
        info["classes"] = list(names.values()) if isinstance(names, dict) else names
    for split in ("train", "valid", "val", "test"):
        img_dir = path / split / "images"
        if not img_dir.is_dir():
            continue
        images = [f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXT]
        lbl_dir = path / split / "labels"
        labels = list(lbl_dir.glob("*.txt")) if lbl_dir.is_dir() else []
        info["splits"][split] = {"images": len(images), "labels": len(labels)}
    return info


summary, counts = [], []
for r in rows:
    info = inspect_dataset(ROOT / r["found_at"]) if r["found_at"] else inspect_dataset(resolve(r["location"]))
    summary.append({
        "dataset": r["dataset"],
        "location": r["found_at"] or r["location"],
        "files": info["files"],
        "classes": info["classes"],
    })
    for split, n in info["splits"].items():
        counts.append({
            "dataset": r["dataset"],
            "split": split,
            "images": n["images"],
            "labels": n["labels"],
            "unlabeled": n["images"] - n["labels"],
        })

display(pd.DataFrame(summary))
counts = pd.DataFrame(counts, columns=["dataset", "split", "images", "labels", "unlabeled"])
display(counts)

if counts.empty:
    print("No YOLO-structured datasets found on disk yet.")
else:
    counts.pivot_table(
        index="dataset", columns="split", values="images", aggfunc="sum", fill_value=0
    ).plot(kind="bar", figsize=(8, 4), title="Images per split")
    plt.tight_layout()
    plt.show()

## 4. Config health checks

| check | catches |
|---|---|
| `missing_location` | `dataset_location` that does not exist on disk |
| `stale_base_dir` | location not under `data/` (e.g. leftover `datasets/` paths) |
| `location_task_mismatch` | location that does not contain its config's folder name |
| `task_name_mismatch` | `task_name` differing from the config's folder name |
| `duplicate_source` | same Roboflow project + version in more than one entry |
| `duplicate_location` | two entries writing to the same directory |
| `name_reused` | one `dataset_name` pointing at different sources |

In [ ]:
issues = []

for r in rows:
    parts = Path(r["location"]).parts
    if not r["on_disk"]:
        hint = f" (but found at {r['found_at']})" if r["found_at"] else ""
        issues.append(("missing_location", f"{r['dataset']}: {r['location']} does not exist{hint}"))
    if parts and parts[0] != DATA_DIR:
        issues.append(("stale_base_dir", f"{r['dataset']}: {r['location']} is not under {DATA_DIR}/"))
    if r["config"] not in parts:
        issues.append((
            "location_task_mismatch",
            f"{r['dataset']}: config folder is '{r['config']}' but location is {r['location']}",
        ))

for c in CONFIGS:
    if c["kind"] == "invalid":
        issues.append(("invalid_json", f"{c['path'].relative_to(ROOT)}: {c['error']}"))
    elif c["kind"] == "bronze" and c["cfg"].get("task_name") != c["path"].parent.name:
        issues.append((
            "task_name_mismatch",
            f"{c['path'].relative_to(ROOT)}: task_name='{c['cfg'].get('task_name')}'",
        ))

if not bronze.empty:
    for (ref, version), g in bronze[bronze.duplicated(["ref", "version"], keep=False)].groupby(["ref", "version"]):
        issues.append(("duplicate_source", f"{ref} v{version} appears in: {sorted(g['config'])}"))
    for loc, g in bronze[bronze.duplicated("location", keep=False)].groupby("location"):
        issues.append(("duplicate_location", f"{loc} used by: {sorted(g['dataset'])}"))
    ref_counts = bronze.groupby("dataset")["ref"].nunique()
    for name in ref_counts[ref_counts > 1].index:
        issues.append(("name_reused", f"'{name}' points at: {sorted(bronze[bronze['dataset'] == name]['ref'].unique())}"))

issues_df = pd.DataFrame(sorted(set(issues)), columns=["check", "detail"])
if issues_df.empty:
    print("No issues found.")
else:
    display(issues_df)

## 5. Data layers on disk

In [ ]:
layer_rows = []
data_root = ROOT / DATA_DIR
if data_root.is_dir():
    for task_dir in sorted(p for p in data_root.iterdir() if p.is_dir()):
        for layer in sorted(p for p in task_dir.iterdir() if p.is_dir()):
            layer_rows.append({
                "task": task_dir.name,
                "layer": layer.name,
                "entries": sorted(p.name for p in layer.iterdir()),
                "files": sum(1 for f in layer.rglob("*") if f.is_file()),
            })
display(pd.DataFrame(layer_rows, columns=["task", "layer", "entries", "files"]))

## 6. Task registry and silver DAGs

In [ ]:
try:
    from src.transforms.tasks.task_registry import TASK_REGISTRY
except Exception as exc:
    TASK_REGISTRY = {}
    print("could not import TASK_REGISTRY:", repr(exc))

display(pd.DataFrame({
    "type": list(TASK_REGISTRY),
    "class": [cls.__name__ for cls in TASK_REGISTRY.values()],
}))

In [ ]:
def silver_issues(cfg, name):
    found = []
    for ds in cfg["datasets"]:
        ids = {n["id"] for n in ds["chain"]}
        for n in ds["chain"]:
            if n["type"] not in TASK_REGISTRY:
                found.append((name, f"{ds['name']}:{n['id']} uses unregistered type '{n['type']}'"))
            for dep in n.get("depends_on", []):
                if dep not in ids:
                    found.append((name, f"{ds['name']}:{n['id']} depends on unknown node '{dep}'"))
    for key in ("merge", "output"):
        kind = cfg.get(key, {}).get("type")
        if kind not in TASK_REGISTRY:
            found.append((name, f"{key} type '{kind}' is not registered"))
    return found


def build_graph(cfg):
    g = nx.DiGraph()
    ends = []
    for ds in cfg["datasets"]:
        ids = {n["id"] for n in ds["chain"]}
        for n in ds["chain"]:
            nid = f"{ds['name']}:{n['id']}"
            g.add_node(nid, label=f"{n['id']}\n{n['type']}", type=n["type"])
            for dep in n.get("depends_on", []):
                if dep in ids:  # unknown deps are reported by silver_issues, not drawn
                    g.add_edge(f"{ds['name']}:{dep}", nid)
        ends.append(f"{ds['name']}:{ds['chain'][-1]['id']}")
    g.add_node("merge", label=f"merge\n{cfg['merge']['type']}", type=cfg["merge"]["type"])
    g.add_node("output", label=f"output\n{cfg['output']['type']}", type=cfg["output"]["type"])
    for end in ends:
        g.add_edge(end, "merge")
    g.add_edge("merge", "output")
    return g


def draw_graph(g, title):
    for layer, nodes in enumerate(nx.topological_generations(g)):
        for node in nodes:
            g.nodes[node]["layer"] = layer
    depth = max(nx.get_node_attributes(g, "layer").values()) + 1
    pos = nx.multipartite_layout(g, subset_key="layer")
    colors = ["#9fd8a5" if g.nodes[n]["type"] in TASK_REGISTRY else "#f4a3a3" for n in g]
    plt.figure(figsize=(max(8, 2.2 * depth), 5))
    nx.draw(
        g, pos, node_color=colors, node_size=2600,
        labels=nx.get_node_attributes(g, "label"), font_size=7, arrows=True,
    )
    plt.title(f"{title}  (green = registered, red = unregistered)")
    plt.show()


silver_configs = [c for c in CONFIGS if c["kind"] == "silver"]
if not silver_configs:
    print("No silver configs found (a config whose datasets define a 'chain').")
for c in silver_configs:
    name = str(c["path"].relative_to(ROOT))
    for issue in silver_issues(c["cfg"], name):
        print("ISSUE", *issue)
    draw_graph(build_graph(c["cfg"]), name)